# Dimensionality Reduction with PCA (Beginner Version)
**Week 4, Session 2 | AI and Machine Learning Track**

### By the end of this session you will be able to:
1. Say in simple words what dimensionality reduction is
2. Use PCA to turn many columns into fewer columns
3. Check how much information was kept

## 1. The big idea in plain words
Imagine a student report card with **many subjects**. Writing everything is long, so a teacher might summarise it with just **2 numbers**, like "Science score" and "Arts score".

We lose a little detail, but we keep the **main story**. This is exactly what **PCA** does with data.

- Each **column (feature)** in a dataset is called a **dimension**.
- **Dimensionality reduction** = turning **many columns** into **fewer columns**, while keeping the important information.
- **PCA** (Principal Component Analysis) is the most popular way to do it.

**Why do it?** Fewer columns means the data is easier to **plot**, easier to **understand**, and models can run **faster**.

## 2. Import the tools

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

## 3. A tiny example: 2 columns become 1
Here are 8 students with two columns: **hours studied** and **exam score**.

In [ ]:
students = pd.DataFrame({
    "hours_studied": [1, 2, 3, 4, 5, 6, 7, 8],
    "exam_score":    [35, 42, 50, 55, 62, 70, 78, 85],
})
students

In [ ]:
plt.scatter(students["hours_studied"], students["exam_score"])
plt.xlabel("Hours studied")
plt.ylabel("Exam score")
plt.title("The two columns move together")
plt.show()

**Look at the plot.** The two columns go up together, so they carry almost the **same information**. We can summarise both with **one number**. Let's do it with PCA.

**The 3 steps we always follow:**
1. **Scale** the data (put columns on the same range)
2. **Create** the PCA and say how many columns we want
3. **Transform** the data

In [ ]:
# Step 1: scale
scaler = StandardScaler()
students_scaled = scaler.fit_transform(students)

# Step 2: create PCA that gives us 1 column
pca = PCA(n_components=1)

# Step 3: transform
students_pca = pca.fit_transform(students_scaled)

print("Before:", students_scaled.shape)
print("After: ", students_pca.shape)

In [ ]:
# Look at the new single column
pd.DataFrame(students_pca, columns=["PC1"]).round(2)

We went from **2 columns to 1 column** (called **PC1**, short for *Principal Component 1*).

How much information did we keep?

In [ ]:
kept = pca.explained_variance_ratio_[0] * 100
print(f"PC1 keeps {kept:.1f}% of the information")

Because the two columns were so similar, **one column keeps almost everything**.

## 4. A real example: Iris flowers
The Iris dataset has measurements of **150 flowers** from **3 species**. Each flower has **4 columns**.

In [ ]:
iris = load_iris(as_frame=True)
flowers = iris.data             # the 4 measurement columns
species = iris.target           # 0, 1, 2 for the 3 species

flowers.head()

We can't draw a plot with 4 dimensions. Let's use PCA to reduce **4 columns to 2** so we can see the flowers on a normal graph.

In [ ]:
# Step 1: scale
flowers_scaled = StandardScaler().fit_transform(flowers)

# Step 2: create PCA with 2 columns
pca = PCA(n_components=2)

# Step 3: transform
flowers_pca = pca.fit_transform(flowers_scaled)

print("Before:", flowers_scaled.shape)
print("After: ", flowers_pca.shape)

In [ ]:
# Put the result in a table with clear names
flowers_2d = pd.DataFrame(flowers_pca, columns=["PC1", "PC2"])
flowers_2d["species"] = species.map({0: "setosa", 1: "versicolor", 2: "virginica"})
flowers_2d.head()

In [ ]:
# Plot each species with its own colour
for name, colour in [("setosa", "red"), ("versicolor", "green"), ("virginica", "blue")]:
    group = flowers_2d[flowers_2d["species"] == name]
    plt.scatter(group["PC1"], group["PC2"], color=colour, label=name)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Iris flowers: 4 columns reduced to 2")
plt.legend()
plt.show()

**Discussion:** PCA never saw the species names, yet the species form separate groups. Which species is easiest to tell apart from the others?

## 5. How much information did we keep?

In [ ]:
ratios = pca.explained_variance_ratio_

print(f"PC1 keeps {ratios[0] * 100:.1f}% of the information")
print(f"PC2 keeps {ratios[1] * 100:.1f}% of the information")
print(f"Together they keep {ratios.sum() * 100:.1f}%")

We dropped from 4 columns to 2 and still kept most of the information. That is the power of PCA.

## 6. How many columns should we keep?
Let's keep **all 4** components and see how the information adds up as we add more of them.

In [ ]:
pca_all = PCA()                       # no number given = keep all 4
pca_all.fit(flowers_scaled)

running_total = pca_all.explained_variance_ratio_.cumsum() * 100

for i, total in enumerate(running_total, start=1):
    print(f"Using {i} component(s) keeps {total:.1f}% of the information")

In [ ]:
plt.plot([1, 2, 3, 4], running_total, marker="o")
plt.xlabel("Number of components")
plt.ylabel("Information kept (%)")
plt.title("How much information do we keep?")
plt.xticks([1, 2, 3, 4])
plt.grid(alpha=0.3)
plt.show()

**Rule of thumb:** keep enough components to hold about **90% to 95%** of the information. Here, **2 components** is already enough.

## 7. Quick summary: the PCA recipe
```python
scaled = StandardScaler().fit_transform(data)   # 1. scale
pca = PCA(n_components=2)                       # 2. choose how many columns
reduced = pca.fit_transform(scaled)             # 3. transform
pca.explained_variance_ratio_                   # 4. check information kept
```

| Word | Simple meaning |
|---|---|
| **Dimension** | A column (feature) in the data |
| **PCA** | A method that turns many columns into fewer |
| **Principal Component (PC)** | One of the new columns PCA creates |
| **Explained variance ratio** | How much information a component keeps |

## Practice exercises
1. Change `n_components` to **3** in the Iris example. How much information is kept now?
2. Change `n_components` to **1**. Plot the result. Can you still separate the species?
3. In the tiny student example, add a third column called `hours_slept` with your own numbers. Reduce 3 columns to 2.
4. **Challenge:** run PCA on Iris **without scaling**. Do the percentages change?

## Key takeaways
- **Dimensionality reduction** means fewer columns, same main story.
- **PCA** creates new columns called **principal components**.
- Always **scale** the data first.
- Check **how much information is kept** (aim for about 90% to 95%).